In [3]:
import os
os.environ["KERAS_BACKEND"] = "torch"
import torch
from torch import nn
import keras
from keras.layers import TorchModuleWrapper
from torch.nn import functional as F
import numpy as np

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [5]:
# Define the dimensions based on the model
batch_size = 64
input_features = 32
num_classes = 10
# 1. Generate random input data (x)
# Shape: (batch_size, input_features) -> e.g., (64, 32)
# Using float32 is standard for deep learning frameworks.
x_test = np.random.rand(batch_size, input_features).astype('float32')

# 2. Generate random integer labels (y)
# Shape: (batch_size,) -> e.g., (64,)
# The labels are integers from 0 to 9 for 10 classes.
y_test = np.random.randint(0, num_classes, size=batch_size)

print("Shape of x_test:", x_test.shape)
print("Shape of y_test:", y_test.shape)
print("Sample labels:", y_test[:5])

Shape of x_test: (64, 32)
Shape of y_test: (64,)
Sample labels: [1 1 4 9 8]


In [10]:
import keras
import numpy as np

## 1. Définition du modèle avec une connexion résiduelle
# On utilise l'API fonctionnelle de Keras 3.

# Définit la forme de l'entrée (ex: une image de 32x32 pixels avec 3 canaux de couleur)
inputs = keras.Input(shape=(3, 32, 32))

# --- Début du bloc résiduel ---
# Le chemin principal traite les données.
# On utilise une couche de convolution qui conserve les dimensions de l'image
# (filters=3 pour garder 3 canaux, padding='same' pour garder la taille 32x32).
fx = keras.layers.Conv2D(filters=3, kernel_size=(3, 3), padding='same', activation='relu')(inputs)

# La connexion résiduelle est simplement l'entrée originale.
# On ajoute la sortie du chemin principal (fx) avec l'entrée originale (inputs).
# Les formes doivent être identiques pour que l'addition fonctionne !
block_output = keras.layers.Add()([fx, inputs])
# --- Fin du bloc résiduel ---


# Applique une couche de pooling à la sortie du bloc résiduel.
outputs = keras.layers.GlobalAveragePooling2D()(block_output)

# Crée le modèle en liant l'entrée à la sortie finale.
model = keras.Model(inputs=inputs, outputs=outputs)


## 2. Affichage du résumé du modèle
# La méthode summary() montre l'architecture et la forme des données à chaque étape.
print("Résumé du modèle avec connexion résiduelle :")
model.summary()


## 3. Exemple d'utilisation
# Créons une fausse image pour voir l'effet du pooling.
dummy_image = np.random.rand(1, 3, 32, 32) # Batch de 1 image

# Faisons une prédiction
output_tensor = model.predict(dummy_image)

print("\n--- Exemple ---")
print(f"Forme de l'image d'entrée : {dummy_image.shape}")
print(f"Forme du tenseur de sortie : {output_tensor.shape}")

Résumé du modèle avec connexion résiduelle :


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_4       │ (None, 3, 32, 32) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 3, 32, 32) │         84 │ input_layer_4[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_4 (Add)         │ (None, 3, 32, 32) │          0 │ conv2d_4[0][0],   │
│                     │                   │            │ input_layer_4[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 3)         │          0 │ add_4[0][0]       │
│ (GlobalAveragePool… │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 84 (336.00 B)

 Trainable params: 84 (336.00 B)

 Non-trainable params: 0 (0.00 B)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 173ms/step

--- Exemple ---
Forme de l'image d'entrée : (1, 3, 32, 32)
Forme du tenseur de sortie : (1, 3)


In [11]:
from decomon.layers import DecomonLayer
from decomon.models import clone
from decomon.perturbation_domain import BallDomain
from decomon import get_lower_noise, get_range_noise, get_upper_noise

In [12]:
perturbation_domain = BallDomain(eps=1, p=2)
decomon_model = clone(model,final_ibp=True, final_affine=False, perturbation_domain=perturbation_domain, method='crown')
lower, upper = decomon_model.predict(dummy_image, verbose=0)

2025-10-07 15:31:04.292856: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1759843864.320741   25914 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1759843864.329054   25914 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-10-07 15:31:04.355846: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/home/aws_install/miniconda3/envs/k3torchenv/lib/python3.10/site-packages/keras/src/models/functional.py:238: UserWarning: Th

In [13]:
lower

array([[0.5605405 , 0.47682554, 0.8229533 ]], dtype=float32)

In [14]:
upper

array([[0.9751956, 0.7526674, 1.2506901]], dtype=float32)